Si consideri il data set housing.csv, allegato al presente compito, costituito da
20640 record che descrivono il valore medio di uno stabile in California sulla base delle
seguenti caratteristiche:
•longitude
• latitude
• housing_median_age
• total_rooms (numero totale di
camere nello stabile)
• total_bedrooms (numero
totale di camera da letto nello
stabile)
• population (numero di abitanti
dello stabile)
• households (numero di
famiglie nello stabile)
• median_income
• median_house_value
• ocean_proximity
1. Individuare eventuali dati mancanti, farne l’imputazione e procedere al
trattamento delle feature categoriche.
punti ___/ 4
2. Eseguire la feature selection per individuare le caratteristiche più rilevanti del
data set attraverso una tecnica embedded che impieghi un regressore come
modello. Infatti, sappiamo che la regressione lineare attribuisce coefficienti
piccoli o nulli alle feature che influenzano poco la predizione.
punti ___/ 8
3. Implementare una regressione Lasso ed una Ridge per il data set così
processato e confrontarne le prestazioni in termini di RMSE e di R2.
punti ___/ 8
4. Implementare una piccola rete neurale convoluzionale in Tensorflow che
esegua la regressione e confrontare i risultati con i modelli precedenti usando
sempre RMSE e R2.

In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd 

df = pd.read_csv('housing.csv')

df = df.dropna( subset=['median_house_value'])

X = df.drop( columns=['median_house_value'] )
y = df.median_house_value

X_tr, X_te, y_tr, y_te = train_test_split(X,y, test_size=0.15, random_state=42)


In [2]:
from sklearn.impute import SimpleImputer

num_f = X_tr.select_dtypes(include=['number']).columns.tolist()
cat_f = [ col for col in X_tr.columns if col not in num_f ]

imputer1 = SimpleImputer(strategy='median').set_output(transform='pandas')
imputer2 = SimpleImputer(strategy='constant').set_output(transform='pandas')

X_tr_num = imputer1.fit_transform( X_tr[num_f] )
X_te_num = imputer1.transform( X_te[num_f] )

X_tr_cat = imputer2.fit_transform( X_tr[cat_f] )
X_te_cat = imputer2.transform( X_te[cat_f] )

In [3]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder().set_output(transform='pandas')


X_tr_cat = encoder.fit_transform(X_tr_cat)
X_te_cat = encoder.transform(X_te_cat)


X_tr = pd.concat( [X_tr_num, X_tr_cat] , axis=1 )

X_te = pd.concat( [X_te_num, X_te_cat] , axis=1 )

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().set_output(transform='pandas')

X_tr = scaler.fit_transform( X_tr )
X_te= scaler.transform( X_te )



In [ ]:

from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LassoCV, Lasso, Ridge
from sklearn.metrics import r2_score, root_mean_squared_error

# ==========================================
# COMPLEMENTO PUNTO 2: SELEZIONE EMBEDDED
# ==========================================
# 1. Configura il selettore basato sui coefficienti lineari
selector_model = LassoCV(random_state=42, cv=3)
sel = SelectFromModel(estimator=selector_model)
print(X_tr)
# 2. Applica la selezione sui dataset
X_tr_sel = sel.fit_transform(X_tr, y_tr)
X_te_sel = sel.transform(X_te)

print(f"PUNTO 2 -> Feature selezionate dal modello embedded: {X_tr_sel.shape[1]}")


# ==========================================
# COMPLEMENTO PUNTO 3: ADDESTRAMENTO E CONFRONTO
# ==========================================

# 1. Modello LASSO
lasso = Lasso(alpha=1.0, random_state=42)
lasso.fit(X_tr_sel, y_tr)
y_pred_lasso = lasso.predict(X_te_sel)

r2_lasso = r2_score(y_te, y_pred_lasso)
rmse_lasso = root_mean_squared_error(y_te, y_pred_lasso)

# 2. Modello RIDGE
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_tr_sel, y_tr)
y_pred_ridge = ridge.predict(X_te_sel)

r2_ridge = r2_score(y_te, y_pred_ridge)
rmse_ridge = root_mean_squared_error(y_te, y_pred_ridge)

# 3. Creazione della tabella riassuntiva per il professore
df_confronto = pd.DataFrame({
    'Metrica di Valutazione': ['R² Score (Più alto è meglio)', 'RMSE (Più basso è meglio)'],
    'Regressione LASSO': [f"{r2_lasso:.4f}", f"{rmse_lasso:.4f}"],
    'Regressione RIDGE': [f"{r2_ridge:.4f}", f"{rmse_ridge:.4f}"]
})

print("\n--- PUNTO 3: TABELLA DI CONFRONTO FINALE ---")
print(df_confronto.to_string(index=False))

       longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
4602    0.654023 -0.745627           -1.315284    -0.967228       -0.502063   
6978    0.773694 -0.783067            0.271015    -0.078305        0.033339   
16415  -0.836879  1.046825            1.064165    -0.999418       -1.063758   
2549   -2.287891  2.408714            1.143480    -0.162461       -0.141145   
11025   0.878406 -0.867307           -0.204874     0.001252        0.226944   
...          ...       ...                 ...          ...             ...   
11284   0.808598 -0.871987            0.508960    -0.601634       -0.805617   
11964   1.072871 -0.759667            0.350330     0.204973        0.076362   
5390    0.599173 -0.754987            0.588275    -0.247076        0.073972   
860    -1.185920  0.906424           -1.077339     0.430767        0.140898   
15795  -1.415290  0.995345            1.857315     0.730141        1.857053   

       population  households  median_income  ocean